# IBGE Municipality Population - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim, split, lower, translate

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.ibge_municipality_population"
target_table = f"{catalog}.silver.ibge_municipality_population"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

In [0]:
display(bronze_df.limit(10))

In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

In [0]:
for column, dtype in bronze_df.dtypes[:2]+bronze_df.dtypes[4:6]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

In [0]:
bronze_df.select("ibge_municipality_id", "year").distinct().count()

- ibge_municipality_id + year columns form a composite key for this table.
- There are no null values or extra whitespace in any column.

In [0]:
display(bronze_df.select('municipality_name').limit(10))

municipality name contains state code. These need to be separated.

## Transform to Silver

In [0]:
silver_df = bronze_df.withColumnRenamed("municipality_name", "municipality_name_raw")

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "municipality_name",
        split(col("municipality_name_raw"), ' \\(')[0]
        )
    .withColumn(
        "state_code",
        split(split(col("municipality_name_raw"), ' \\(')[1], '\\)')[0]
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "municipality_name",
    translate(
        lower(col("municipality_name")),
        "áàâãäéèêëíìîïóòôõöúùûüç-'",
        "aaaaaeeeeiiiiooooouuuuc  "
    )
)

In [0]:
silver_df = silver_df.withColumn("year", col("year").cast("int"))

silver_df = (
    silver_df
    .filter(
        col("population")
        .rlike("^[0-9]+$")
        )
    .withColumn(
        "population",
        col("population").cast("int"))
)

In [0]:
display(silver_df.select('municipality_name', "state_code", "year", "population").limit(20))

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

In [0]:
display(silver_table_df.limit(10))

In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Silver contains 16,710 rows. Three Bronze rows for "boa esperanca do norte" are excluded because IBGE returned "..." instead of numeric values. The original source values are preserved in Bronze.